# CaPy v2 — Training & Evaluation

Tri-modal contrastive learning for drug discovery.

**Phases:**
- **Phase 1:** Bi-modal mol↔morph baseline (gate: compound R@10 > 10%) — PASSED
- **Phase 2:** Tri-modal + bi-modal baselines + remediation sweeps — PASSED
- **Phase 3:** Full ablation matrix (4 configs × 5 seeds) + evaluation report

**Runtime:** GPU (H100 recommended, T4/V100 also work)

In [ ]:
# Clone repo and install (idempotent — safe to re-run)
import os
if not os.path.exists("/content/CaPy-v2"):
    !git clone https://github.com/ogngnaoh/CaPy-v2.git
else:
    %cd /content/CaPy-v2
    !git pull
    %cd /content

%cd /content/CaPy-v2
!pip install -e ".[dev]" -q

# Verify install
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download data (morphology, expression, metadata)
!python3 scripts/download.py

In [ ]:
# Preprocess: QC, normalize, scaffold split
!python3 scripts/preprocess.py

In [ ]:
# Quick sanity check — verify processed data
import json
from pathlib import Path

import pandas as pd

processed = Path("data/processed")
for split in ["train", "val", "test"]:
    df = pd.read_parquet(processed / f"{split}.parquet")
    print(f"{split}: {len(df)} treatments, {df.shape[1]} columns")

with open(processed / "feature_columns.json") as f:
    cols = json.load(f)
print(f"\nMorph features: {len(cols['morph_features'])}")
print(f"Expr features: {len(cols['expr_features'])}")

In [ ]:
# Optional: login to W&B for experiment tracking
# Skip this cell if you don't want W&B logging
import wandb
wandb.login()

## Phase 1: Bi-Modal Mol↔Morph Baseline

**Gate criteria:** compound-level R@10 > 10% AND alignment < 1.5

Run single seed first to verify, then multi-seed for robustness.

In [ ]:
# Phase 1 gate: bi-modal mol-morph (single seed)
!python3 scripts/train.py model=bi_mol_morph seed=42

## Multi-Seed Validation

Run 3 additional seeds to confirm Phase 1 gate result is robust.
All seeds should pass: compound R@10 > 10% AND alignment < 1.5.

In [ ]:
# Multi-seed validation (seeds 123, 456, 789)
for seed in [123, 456, 789]:
    print(f"\n{'='*60}")
    print(f"Training seed={seed}")
    print(f"{'='*60}")
    !python3 scripts/train.py model=bi_mol_morph seed={seed}

In [ ]:
# Compare multi-seed results
import torch

print(f"{'Seed':<8} {'Epoch':<8} {'compound R@10':<16} {'Alignment':<12} {'Uniform mol':<14} {'Uniform morph':<14} {'Gate'}")
print("-" * 90)

for seed in [42, 123, 456, 789]:
    path = f"checkpoints/bi_mol_morph_seed{seed}.pt"
    try:
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        metrics = ckpt.get("metrics", {})
        r10 = ckpt.get("best_metric", metrics.get("compound/mean_R@10", -1))
        align = metrics.get("align_mol_morph", -1)
        u_mol = metrics.get("uniform_mol", -1)
        u_morph = metrics.get("uniform_morph", -1)
        epoch = ckpt.get("epoch", -1)
        gate = "PASS" if (r10 > 0.10 and align < 1.5) else "FAIL"
        print(f"{seed:<8} {epoch:<8} {r10:<16.4f} {align:<12.4f} {u_mol:<14.4f} {u_morph:<14.4f} {gate}")
    except FileNotFoundError:
        print(f"{seed:<8} not found (not trained yet)")

print("\nGate: compound R@10 > 10% AND alignment < 1.5")

## Phase 2: Additional Bi-Modal Baselines + Tri-Modal

After multi-seed validation passes, train remaining baselines and tri-modal.

In [ ]:
# B5: Mol↔Expr bi-modal
!python3 scripts/train.py model=bi_mol_expr seed=42

In [ ]:
# B6: Morph↔Expr bi-modal
!python3 scripts/train.py model=bi_morph_expr seed=42

In [ ]:
# T1: Tri-modal (all 3 modalities)
!python3 scripts/train.py model=tri_modal seed=42

In [ ]:
# Compare all configs (single seed)
import torch

configs = [
    ("bi_mol_morph", 42),
    ("bi_mol_expr", 42),
    ("bi_morph_expr", 42),
    ("tri_modal", 42),
]

print(f"{'Config':<20} {'Epoch':<8} {'Best R@10':<12}")
print("-" * 42)

for config, seed in configs:
    path = f"checkpoints/{config}_seed{seed}.pt"
    try:
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        r10 = ckpt.get("best_metric", -1)
        epoch = ckpt.get("epoch", -1)
        print(f"{config:<20} {epoch:<8} {r10:<12.4f}")
    except FileNotFoundError:
        print(f"{config:<20} not found")

## Full Ablation Matrix (5 seeds)

Run the complete 20-run matrix (4 trained configs x 5 seeds) for statistical analysis.

In [ ]:
# Full ablation matrix with resume support
# --resume skips configs that already have checkpoints
!python3 scripts/run_ablations.py --matrix core --resume --configs B4 B5 B6 T1

## Phase 2 Remediation: Mol-Direction Sweeps

T1 morph↔expr R@10 = 84.8% (beats B6 by +10pp), but mol-containing directions stuck at ~12%.
These sweeps test infrastructure to push mol R@10 toward 15-20%+.

In [ ]:
# Sweep 1: Per-pair SigLIP baseline (equal weights, isolates effect of separate temp/bias)
!python3 scripts/train.py model=tri_modal seed=42

In [ ]:
# Sweep 2a: Downweight morph↔expr
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.pair_weights.mol_morph=1 \
    training.loss.pair_weights.mol_expr=1 \
    training.loss.pair_weights.morph_expr=0.3

In [ ]:
# Sweep 2b: Upweight mol pairs 2x
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.pair_weights.mol_morph=2 \
    training.loss.pair_weights.mol_expr=2 \
    training.loss.pair_weights.morph_expr=1

In [ ]:
# Sweep 2c: Upweight mol pairs 3x, downweight morph↔expr
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.pair_weights.mol_morph=3 \
    training.loss.pair_weights.mol_expr=3 \
    training.loss.pair_weights.morph_expr=0.5

In [ ]:
# Sweep 3a: Discriminative LR — mol 2x, morph/expr 0.5x
!python3 scripts/train.py model=tri_modal seed=42 \
    training.optimizer.modality_lr_mult.mol=2.0 \
    training.optimizer.modality_lr_mult.morph=0.5 \
    training.optimizer.modality_lr_mult.expr=0.5

In [ ]:
# Sweep 3b: Discriminative LR — mol 3x, morph/expr 0.3x
!python3 scripts/train.py model=tri_modal seed=42 \
    training.optimizer.modality_lr_mult.mol=3.0 \
    training.optimizer.modality_lr_mult.morph=0.3 \
    training.optimizer.modality_lr_mult.expr=0.3

In [ ]:
# Sweep 4: Curriculum — ramp mol pair weights from 0 to 1.0 over 50 epochs
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.curriculum.enabled=true \
    training.loss.curriculum.warmup_epochs=50

In [ ]:
# Sweep 5: Staged — morph↔expr for 100 epochs, then freeze + add mol
!python3 scripts/train.py model=tri_modal training=staged seed=42

In [ ]:
# Compare all sweep results — per-direction R@10
!python3 scripts/analyze_checkpoints.py

## Phase 3: Full Ablation Matrix (8 configs × 5 seeds = 40 runs)

**S2b locked as default T1 config** (per-pair SigLIP + 2x mol pair weights).

The ablation harness trains B4, B5, B6, T1 × 5 seeds each (20 trained runs).
B0–B3 are raw-feature baselines (skipped automatically). `--resume` skips
any seed/config combos that already have checkpoints from earlier phases.

**Estimated runtime:** ~20 runs × ~15 min each ≈ 5 hours on H100.

In [ ]:
# Phase 3: Full ablation matrix
# --resume skips configs with existing checkpoints (reuses Phase 1/2 seed=42 runs)
# B0-B3 (raw-feature baselines) are skipped automatically (no training needed)
!python3 scripts/run_ablations.py --matrix core --resume

In [ ]:
# Generate ablation summary with statistical analysis (FR-9.2)
# Outputs: results/ablation_summary.csv, results/ablation_comparison.tex,
#          results/ablation_barplot.png, plus stdout comparison table
!python3 -m scripts.summarize_ablations

In [ ]:
# Collect and display all ablation results
import json
from pathlib import Path

import torch

configs = {
    "B4": "bi_mol_morph",
    "B5": "bi_mol_expr",
    "B6": "bi_morph_expr",
    "T1": "tri_modal",
}
seeds = [42, 123, 456, 789, 1024]

# Header
directions = ["mol→morph", "morph→mol", "mol→expr", "expr→mol", "morph→expr", "expr→morph"]
print(f"{'Config':<6} {'Seed':<6} {'Epoch':<6} {'mean R@10':<10} {'cmpd R@10':<10} " + " ".join(f"{d:<12}" for d in directions))
print("-" * 120)

for config_id, name in configs.items():
    for seed in seeds:
        path = Path(f"checkpoints/{name}_seed{seed}.pt")
        if not path.exists():
            print(f"{config_id:<6} {seed:<6} {'—'}")
            continue
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        m = ckpt.get("metrics", {})
        epoch = ckpt.get("epoch", -1)
        mean_r10 = m.get("mean_R@10", -1)
        cmpd_r10 = ckpt.get("best_metric", m.get("compound/mean_R@10", -1))

        dir_vals = []
        for pair in [("mol", "morph"), ("morph", "mol"), ("mol", "expr"), ("expr", "mol"), ("morph", "expr"), ("expr", "morph")]:
            key = f"compound/{pair[0]}->{pair[1]}/R@10"
            val = m.get(key, m.get(f"{pair[0]}->{pair[1]}/R@10", -1))
            dir_vals.append(f"{val:<12.4f}" if val >= 0 else f"{'N/A':<12}")

        print(f"{config_id:<6} {seed:<6} {epoch:<6} {mean_r10:<10.4f} {cmpd_r10:<10.4f} " + " ".join(dir_vals))
    print()

# Also show ablation_runs.jsonl if it exists
jsonl = Path("results/ablation_runs.jsonl")
if jsonl.exists():
    print(f"\nResults also saved to: {jsonl}")
    print(f"Lines: {sum(1 for _ in open(jsonl))}")

## Phase 3: Full Evaluation Report

Run full evaluation on best T1 checkpoint (FR-8.4).
Generates: retrieval table (CSV + LaTeX), UMAP plots, similarity heatmap,
training curves, MOA clustering metrics, and formatted summary table.

In [ ]:
# Full evaluation report on best T1 checkpoint
# Outputs: results/retrieval_table.csv, results/retrieval_table.tex,
#          results/umap_*.png, results/similarity_heatmap.png,
#          results/training_curves.png, plus MOA clustering + summary table
!python3 scripts/evaluate.py --checkpoint checkpoints/tri_modal_seed42.pt --full \
    --data-dir data/processed --output-dir results